## Импорты

In [485]:
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm
pd.set_option('display.max_columns', 50)

## Пропуски в продажах

In [486]:
games = pd.read_csv('data/games_complete.csv')
games.head()

,id,first_release_date,game_modes,genres,name,platforms,rating,summary,themes,total_rating,total_rating_count,game_type,storyline,age_ratings,game_status,aggregated_rating,aggregated_rating_count,hypes
0,376092,2025-10-31,Single player,Indie,Silly Survivors,PC (Microsoft Windows),100.0,"Silly Survivors - a casual, pared-down rogueli...",Action,100.0,6,0,NaN,NaN,NaN,NaN,NaN,NaN
1,330539,2025-04-18,"Multiplayer, Co-operative, Massively Multiplay...","Shooter, Role-playing (RPG), Simulator",Oxide: Survival Island,"Android, iOS",100.0,OXIDE is a mobile multiplayer survival simulat...,"Survival, Stealth, Sandbox, Open world",100.0,9,0,"Here you are, alone on the abandoned island, w...",NaN,NaN,NaN,NaN,NaN
2,328386,2025-01-16,NaN,NaN,Rooftop Rascal: The Claus Cat,"PlayStation 4, PlayStation 5",100.0,"In ""Rooftop Rascal: The Claus Cat,"" take contr...",NaN,100.0,7,0,NaN,[196045],NaN,NaN,NaN,NaN
3,269473,2023-10-06,"Single player, Multiplayer, Co-operative",Indie,Resonite,"Linux, PC (Microsoft Windows), SteamVR",100.0,Enter a novel digital universe with infinite p...,"Business, Sandbox, Educational",100.0,10,0,NaN,NaN,4.0,NaN,NaN,NaN
4,65755,2011-07-29,Single player,"Point-and-click, Adventure",Trailer Park King,Xbox 360,100.0,"He's a gamer, a bootlegger, a photographer and...",Comedy,100.0,47,0,NaN,NaN,NaN,NaN,NaN,NaN


In [487]:
games[games['name'].str.contains('God')].head()

,id,first_release_date,game_modes,genres,name,platforms,rating,summary,themes,total_rating,total_rating_count,game_type,storyline,age_ratings,game_status,aggregated_rating,aggregated_rating_count,hypes
133,19560,2018-04-20,Single player,"Role-playing (RPG), Hack and slash/Beat 'em up...",God of War,"PlayStation 4, PC (Microsoft Windows)",92.143940,God of War is the sequel to God of War III as ...,"Action, Fantasy, Historical",94.187355,3358,0,"Many years have passed since Kratos, Spartan w...","[63045, 222344, 63046, 23748, 47579, 187740, 6...",NaN,96.230769,26.0,92.0
183,112875,2022-11-09,Single player,"Role-playing (RPG), Hack and slash/Beat 'em up...",God of War Ragnarök,"PlayStation 4, PC (Microsoft Windows), PlaySta...",90.961843,God of War: Ragnarök is the ninth installment ...,"Action, Fantasy, Open world",92.788614,1014,0,The freezing winds of Fimbulwinter have come t...,"[188434, 105532, 112550, 112549, 112548, 11255...",NaN,94.615385,13.0,77.0
310,499,2010-03-16,Single player,"Hack and slash/Beat 'em up, Adventure",God of War III,PlayStation 3,86.153408,"Set in the realm of brutal Greek mythology, Go...","Action, Fantasy, Historical",91.076704,785,0,"Following the ending of God of War II, God of ...","[29125, 222347, 184209, 29126]",NaN,96.000000,6.0,NaN
414,105420,2018-08-23,Single player,"Platform, Hack and slash/Beat 'em up, Adventur...",Hollow Knight: Godmaster,"PlayStation 4, Linux, PC (Microsoft Windows), ...",89.960326,Godmaster was the fourth and final free conten...,"Action, Fantasy",89.960326,30,14,NaN,NaN,NaN,NaN,NaN,1.0
466,9759,2017-09-28,"Single player, Multiplayer","Puzzle, Real Time Strategy (RTS), Strategy, Ad...",Total War: Warhammer II - Serpent God Edition,PC (Microsoft Windows),89.843435,"As the twin-tailed comet arcs across the sky, ...","Action, Fantasy, Comedy",89.843435,12,0,NaN,[15438],NaN,NaN,NaN,NaN


In [488]:
sales = pd.read_csv('vgchartz-1_18_2026.csv')
sales.head()

,img,title,console,genre,publisher,developer,vg_score,critic_score,user_score,total_shipped,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_5741036AmericaFrontccc.jpg,God of War,Series,Action,Sony Interactive Entertainment,SIE Santa Monica Studio,NaN,NaN,NaN,76.55,NaN,NaN,NaN,NaN,NaN,2005-03-22,2020-03-04
1,/games/boxart/full_3351915AmericaFrontccc.jpg,Warriors,Series,Action,KOEI,Omega Force,NaN,NaN,NaN,54.17,NaN,NaN,NaN,NaN,NaN,1997-06-30,2020-03-24
2,/games/boxart/full_6662824AmericaFrontccc.png,Devil May Cry,Series,Action,Capcom,Capcom,NaN,NaN,NaN,37.00,NaN,NaN,NaN,NaN,NaN,2001-10-16,2020-02-03
3,/games/boxart/full_8122622AmericaFrontccc.jpg,God of War (2018),All,Action,Sony Interactive Entertainment,SIE Santa Monica Studio,NaN,NaN,NaN,23.52,NaN,NaN,NaN,NaN,NaN,2018-04-20,2022-11-23
4,/games/boxart/full_6138740AmericaFrontccc.jpg,Dynasty Warriors,Series,Action,KOEI,Omega Force,NaN,NaN,NaN,23.00,NaN,NaN,NaN,NaN,NaN,1997-06-30,2020-03-24


In [489]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 66849 entries, 0 to 66848
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   img            66849 non-null  str    
 1   title          66849 non-null  str    
 2   console        66849 non-null  str    
 3   genre          66849 non-null  str    
 4   publisher      66849 non-null  str    
 5   developer      66832 non-null  str    
 6   vg_score       2470 non-null   float64
 7   critic_score   6780 non-null   float64
 8   user_score     419 non-null    float64
 9   total_shipped  4608 non-null   float64
 10  total_sales    18915 non-null  float64
 11  na_sales       12631 non-null  float64
 12  jp_sales       6722 non-null   float64
 13  pal_sales      12817 non-null  float64
 14  other_sales    15121 non-null  float64
 15  release_date   57719 non-null  str    
 16  last_update    20702 non-null  str    
dtypes: float64(9), str(8)
memory usage: 8.7 MB


Как мы видим маловато данных о продажах(

In [490]:
# Заполним пропуски в total_sales данными из total_shipped
sales['total_sales'] = sales['total_sales'].fillna(sales['total_shipped'])
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 66849 entries, 0 to 66848
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   img            66849 non-null  str    
 1   title          66849 non-null  str    
 2   console        66849 non-null  str    
 3   genre          66849 non-null  str    
 4   publisher      66849 non-null  str    
 5   developer      66832 non-null  str    
 6   vg_score       2470 non-null   float64
 7   critic_score   6780 non-null   float64
 8   user_score     419 non-null    float64
 9   total_shipped  4608 non-null   float64
 10  total_sales    23523 non-null  float64
 11  na_sales       12631 non-null  float64
 12  jp_sales       6722 non-null   float64
 13  pal_sales      12817 non-null  float64
 14  other_sales    15121 non-null  float64
 15  release_date   57719 non-null  str    
 16  last_update    20702 non-null  str    
dtypes: float64(9), str(8)
memory usage: 8.7 MB


In [491]:
sales.head()

,img,title,console,genre,publisher,developer,vg_score,critic_score,user_score,total_shipped,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_5741036AmericaFrontccc.jpg,God of War,Series,Action,Sony Interactive Entertainment,SIE Santa Monica Studio,NaN,NaN,NaN,76.55,76.55,NaN,NaN,NaN,NaN,2005-03-22,2020-03-04
1,/games/boxart/full_3351915AmericaFrontccc.jpg,Warriors,Series,Action,KOEI,Omega Force,NaN,NaN,NaN,54.17,54.17,NaN,NaN,NaN,NaN,1997-06-30,2020-03-24
2,/games/boxart/full_6662824AmericaFrontccc.png,Devil May Cry,Series,Action,Capcom,Capcom,NaN,NaN,NaN,37.00,37.00,NaN,NaN,NaN,NaN,2001-10-16,2020-02-03
3,/games/boxart/full_8122622AmericaFrontccc.jpg,God of War (2018),All,Action,Sony Interactive Entertainment,SIE Santa Monica Studio,NaN,NaN,NaN,23.52,23.52,NaN,NaN,NaN,NaN,2018-04-20,2022-11-23
4,/games/boxart/full_6138740AmericaFrontccc.jpg,Dynasty Warriors,Series,Action,KOEI,Omega Force,NaN,NaN,NaN,23.00,23.00,NaN,NaN,NaN,NaN,1997-06-30,2020-03-24


Ну чуть побольше стало данных о продажах - теперь попробуем смерджить

## Пробуем мерджить

In [492]:
games[games['name'].str.contains('  ')].head()

sales[sales['title'].str.contains('  ')].head()

,img,title,console,genre,publisher,developer,vg_score,critic_score,user_score,total_shipped,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
885,/games/boxart/8438223ccc.jpg,Hunter: The Reckoning Wayward,PS2,Action,VU Games,High Voltage Software,NaN,NaN,NaN,NaN,0.43,0.21,NaN,0.16,0.06,2003-09-09,NaN
1148,/games/boxart/full_9965630AmericaFrontccc.jpg,Warriors Orochi 4,NS,Action,Koei Tecmo,Omega Force,NaN,8.5,NaN,0.31,0.31,NaN,NaN,NaN,NaN,2018-10-16,2018-11-11
1462,/games/boxart/full_2411176AmericaFrontccc.jpg,Warriors Orochi 4,PS4,Action,Koei Tecmo,Omega Force,NaN,NaN,NaN,NaN,0.22,0.05,0.16,NaN,0.01,2018-10-16,2018-11-11
2681,/games/boxart/full_8992245AmericaFrontccc.jpg,Warriors Orochi 4,XOne,Action,Koei Tecmo,Omega Force,NaN,NaN,NaN,NaN,0.04,0.03,NaN,NaN,0.00,2018-10-16,2018-11-11
2707,/games/boxart/full_807279AmericaFrontccc.jpg,Code of Princess EX,NS,Action,Nicalis,Studio Saizensen,6.0,7.2,NaN,NaN,0.03,0.02,0.01,NaN,0.00,2018-07-31,2018-03-08


In [493]:
games['name_clean'] = games['name'].str.lower().str.replace(r'[^a-zа-яё0-9\s]', '', regex=True).str.replace(r'\s+', '_', regex=True)
sales['name_clean'] = sales['title'].str.lower().str.replace(r'[^a-zа-яё0-9\s]', '', regex=True).str.replace(r'\s+', '_', regex=True)


display(games[['name', 'name_clean']].head())
display(sales[['title', 'name_clean']].head())

,name,name_clean
0,Silly Survivors,silly_survivors
1,Oxide: Survival Island,oxide_survival_island
2,Rooftop Rascal: The Claus Cat,rooftop_rascal_the_claus_cat
3,Resonite,resonite
4,Trailer Park King,trailer_park_king


,title,name_clean
0,God of War,god_of_war
1,Warriors,warriors
2,Devil May Cry,devil_may_cry
3,God of War (2018),god_of_war_2018
4,Dynasty Warriors,dynasty_warriors


In [ ]:
matched_names = []
match_scores = []

for name in tqdm(games['name_clean']):
    match = process.extractOne(query=name, choices=sales['name_clean'], scorer=fuzz.token_set_ratio)

    if match:
        matched_names.append(match[0])
        match_scores.append(match[1])
    else:
        matched_names.append(None)
        match_scores.append(None)

fuzz_df = pd.DataFrame({
    'games_name_clean': games['name_clean'],
    'matched_sales_name': matched_names,
    'match_score': match_scores
})
# fuzz_df.to_excel('games_with_matches.xlsx', index=False)

In [495]:
# Разобьем игры на разгные платформы 

games['platforms'] = games['platforms'].str.split(', ')
games_exploded = games.explode('platforms').reset_index(drop=True)
games_exploded.head()

,id,first_release_date,game_modes,genres,name,platforms,rating,summary,themes,total_rating,total_rating_count,game_type,storyline,age_ratings,game_status,aggregated_rating,aggregated_rating_count,hypes,name_clean
0,376092,2025-10-31,Single player,Indie,Silly Survivors,PC (Microsoft Windows),100.0,"Silly Survivors - a casual, pared-down rogueli...",Action,100.0,6,0,NaN,NaN,NaN,NaN,NaN,NaN,silly_survivors
1,330539,2025-04-18,"Multiplayer, Co-operative, Massively Multiplay...","Shooter, Role-playing (RPG), Simulator",Oxide: Survival Island,Android,100.0,OXIDE is a mobile multiplayer survival simulat...,"Survival, Stealth, Sandbox, Open world",100.0,9,0,"Here you are, alone on the abandoned island, w...",NaN,NaN,NaN,NaN,NaN,oxide_survival_island
2,330539,2025-04-18,"Multiplayer, Co-operative, Massively Multiplay...","Shooter, Role-playing (RPG), Simulator",Oxide: Survival Island,iOS,100.0,OXIDE is a mobile multiplayer survival simulat...,"Survival, Stealth, Sandbox, Open world",100.0,9,0,"Here you are, alone on the abandoned island, w...",NaN,NaN,NaN,NaN,NaN,oxide_survival_island
3,328386,2025-01-16,NaN,NaN,Rooftop Rascal: The Claus Cat,PlayStation 4,100.0,"In ""Rooftop Rascal: The Claus Cat,"" take contr...",NaN,100.0,7,0,NaN,[196045],NaN,NaN,NaN,NaN,rooftop_rascal_the_claus_cat
4,328386,2025-01-16,NaN,NaN,Rooftop Rascal: The Claus Cat,PlayStation 5,100.0,"In ""Rooftop Rascal: The Claus Cat,"" take contr...",NaN,100.0,7,0,NaN,[196045],NaN,NaN,NaN,NaN,rooftop_rascal_the_claus_cat


### Мапим платформы для мерджа

In [496]:
platforms = pd.read_csv('data/platforms.csv')
platforms = platforms[['id', 'name']].sort_values(by = 'id')
platforms.head()

,id,name
52,3,Linux
210,4,Nintendo 64
211,5,Wii
62,6,PC (Microsoft Windows)
201,7,PlayStation


In [497]:
pd.Series(sales.console.unique()).to_excel('data/sales_platforms.xlsx', index=False)

In [498]:
## Сопоставим платформы в sales с платформами в games
mapping = {
    'Series': 'Xbox Series X|S',
    'PS4': 'PlayStation 4',
    'PS3': 'PlayStation 3',
    'PS2': 'PlayStation 2',
    'X360': 'Xbox 360',
    'PC': 'PC (Microsoft Windows)',
    'XOne': 'Xbox One',
    'PS': 'PlayStation',
    'PSP': 'PlayStation Portable',
    'Wii': 'Wii',
    'DS': 'Nintendo DS',
    '3DS': 'Nintendo 3DS',
    '2600': 'Atari 2600',
    'NS': 'Nintendo Switch',
    'iOS': 'iOS',
    'NES': 'Nintendo Entertainment System',
    'GC': 'Nintendo GameCube',
    'WiiU': 'Wii U',
    'XB': 'Xbox',
    'N64': 'Nintendo 64',
    'GEN': 'Sega Mega Drive/Genesis',
    'GBA': 'Game Boy Advance',
    'PS5': 'PlayStation 5',
    'GB': 'Game Boy',
    'PSV': 'PlayStation Vita',
    'SNES': 'Super Nintendo Entertainment System',
    'DC': 'Dreamcast',
    'SAT': 'Sega Saturn',
    'GBC': 'Game Boy Color',
    '5200': 'Atari 5200',
    'OSX': 'Mac',
    'And': 'Android',
    'DSiW': 'Nintendo DSi',
    'Lynx': 'Atari Lynx',
    'SCD': 'Sega CD',
    'Linux': 'Linux',
    'MS': 'Sega Master System/Mark III',
    'WW': 'WonderSwan',
    'ZXS': 'ZX Spectrum',
    'ACPC': 'Amstrad CPC',
    'Amig': 'Amiga',
    '7800': 'Atari 7800',
    'GG': 'Sega Game Gear',
    'DSi': 'Nintendo DSi',
    'PCE': 'TurboGrafx-16/PC Engine',
    'AJ': 'Atari Jaguar',
    'WinP': 'Windows Phone',
    'WS': 'WonderSwan',
    'NG': 'Neo Geo AES',
    'GIZ': 'Gizmondo',
    '3DO': '3DO Interactive Multiplayer',
    'VB': 'Virtual Boy',
    'Ouya': 'Ouya',
    'NGage': 'N-Gage',
    'AST': 'Atari ST/STE',
    'S32X': 'Sega 32X',
    'PCFX': 'PC-FX',
    'XS': 'Xbox Series X|S',
    'Int': 'Intellivision',
    'CV': 'ColecoVision',
    'ApII': 'Apple II',
    'NS2': 'Nintendo Switch 2',
    'Arc': 'Arcade',
    'C64': 'Commodore C64/128/MAX',
    'FDS': 'Family Computer Disk System',
    'MSX': 'MSX',
    'C128': 'Commodore C64/128/MAX',
    'CDi': 'Philips CD-i',
    'CD32': 'Amiga CD32',
    'FMT': 'FM Towns',
    'Aco': 'Acorn Archimedes',
    'BBCM': 'BBC Microcomputer System',
    'TG16': 'TurboGrafx-16/PC Engine'
}

map_df = pd.DataFrame(mapping.items(), columns=['sales', 'games'])
map_df


,sales,games
0,Series,Xbox Series X|S
1,PS4,PlayStation 4
2,PS3,PlayStation 3
3,PS2,PlayStation 2
4,X360,Xbox 360
...,...,...
68,CD32,Amiga CD32
69,FMT,FM Towns
70,Aco,Acorn Archimedes
71,BBCM,BBC Microcomputer System


### Мапим названия игр дл мерджа

In [499]:
name_map = pd.read_excel('games_with_matches.xlsx')
name_map.head()

,games_name_clean,matched_sales_name,match_score,flg
0,trailer_park_king,trailer_park_king,100.0,1.0
1,the_legend_of_zelda_tears_of_the_kingdom_ninte...,the_legend_of_zelda_tears_of_the_kingdom_ninte...,100.0,1.0
2,tobal_2,tobal_2,100.0,1.0
3,grand_theft_auto_v,grand_theft_auto_v,100.0,1.0
4,batman_arkham_collection,batman_arkham_collection,100.0,1.0


In [500]:
name_to_map = name_map.loc[name_map['flg'] == 1, ['games_name_clean', 'matched_sales_name']]
name_to_map.head()
games_exploded = games_exploded.merge(name_to_map, how='left',left_on='name_clean', right_on='games_name_clean')

games_exploded['name_clean'] = games_exploded['matched_sales_name'].fillna(games_exploded['name_clean'])

games_exploded = games_exploded.drop(columns=['games_name_clean', 'matched_sales_name'])

### Присоединим all

In [501]:
sales_only_all = sales[sales['console'] == 'All']
sales_only_all.head()

,img,title,console,genre,publisher,developer,vg_score,critic_score,user_score,total_shipped,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update,name_clean
3,/games/boxart/full_8122622AmericaFrontccc.jpg,God of War (2018),All,Action,Sony Interactive Entertainment,SIE Santa Monica Studio,NaN,NaN,NaN,23.52,23.52,NaN,NaN,NaN,NaN,2018-04-20,2022-11-23,god_of_war_2018
7,/games/boxart/full_4571038AmericaFrontccc.jpg,Castle Crashers,All,Action,Microsoft Game Studios,The Behemoth,NaN,NaN,NaN,20.00,20.00,NaN,NaN,NaN,NaN,2008-08-27,2025-12-22,castle_crashers
14,/games/boxart/full_7818120AmericaFrontccc.jpg,God of War: Ragnarök,All,Action,Sony Interactive Entertainment,Sony Computer Entertainment,NaN,NaN,NaN,15.00,15.00,NaN,NaN,NaN,NaN,2022-11-09,2022-11-23,god_of_war_ragnark
15,/games/boxart/full_7549567AmericaFrontccc.jpg,Maneater,All,Action,Tripwire Interactive,Tripwire Interactive,NaN,NaN,NaN,14.00,14.00,NaN,NaN,NaN,NaN,2021-05-25,2024-05-07,maneater
18,/games/boxart/full_6435755AmericaFrontccc.jpg,Devil May Cry 5,All,Action,Capcom,Capcom,NaN,NaN,NaN,12.20,12.20,NaN,NaN,NaN,NaN,2019-03-08,2020-10-11,devil_may_cry_5


In [502]:
games_sales_all = pd.merge(games, sales_only_all, left_on='name_clean', right_on='name_clean', how='inner', suffixes=('_games', '_sales'))
games_sales_all.drop(columns = ['img', 'title', 'console', 'developer', 'release_date', 'last_update'], inplace=True)
games_sales_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 946 entries, 0 to 945
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       946 non-null    int64  
 1   first_release_date       945 non-null    str    
 2   game_modes               943 non-null    str    
 3   genres                   942 non-null    str    
 4   name                     946 non-null    str    
 5   platforms                946 non-null    object 
 6   rating                   939 non-null    float64
 7   summary                  945 non-null    str    
 8   themes                   910 non-null    str    
 9   total_rating             946 non-null    float64
 10  total_rating_count       946 non-null    int64  
 11  game_type                946 non-null    int64  
 12  storyline                472 non-null    str    
 13  age_ratings              906 non-null    str    
 14  game_status              24 non-null 

Уже 900 игр для анализа!!!!

### Примоединим все не all

In [503]:
sales_not_all = sales[sales['console'] != 'All']
sales_not_all.head()

,img,title,console,genre,publisher,developer,vg_score,critic_score,user_score,total_shipped,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update,name_clean
0,/games/boxart/full_5741036AmericaFrontccc.jpg,God of War,Series,Action,Sony Interactive Entertainment,SIE Santa Monica Studio,NaN,NaN,NaN,76.55,76.55,NaN,NaN,NaN,NaN,2005-03-22,2020-03-04,god_of_war
1,/games/boxart/full_3351915AmericaFrontccc.jpg,Warriors,Series,Action,KOEI,Omega Force,NaN,NaN,NaN,54.17,54.17,NaN,NaN,NaN,NaN,1997-06-30,2020-03-24,warriors
2,/games/boxart/full_6662824AmericaFrontccc.png,Devil May Cry,Series,Action,Capcom,Capcom,NaN,NaN,NaN,37.00,37.00,NaN,NaN,NaN,NaN,2001-10-16,2020-02-03,devil_may_cry
4,/games/boxart/full_6138740AmericaFrontccc.jpg,Dynasty Warriors,Series,Action,KOEI,Omega Force,NaN,NaN,NaN,23.00,23.00,NaN,NaN,NaN,NaN,1997-06-30,2020-03-24,dynasty_warriors
5,/games/boxart/full_3094030AmericaFrontccc.jpg,God of War (2018),PS4,Action,Sony Interactive Entertainment,SIE Santa Monica Studio,9.0,9.7,10.0,21.02,21.02,NaN,NaN,NaN,NaN,2018-04-20,2022-05-26,god_of_war_2018


In [504]:
games_ext = games_exploded.merge(map_df, left_on='platforms', right_on='games', how='left')
games_ext.drop(columns=['games'], inplace=True)
games_ext.rename(columns={'sales': 'sales_platform'}, inplace=True)
games_ext.head()

,id,first_release_date,game_modes,genres,name,platforms,rating,summary,themes,total_rating,total_rating_count,game_type,storyline,age_ratings,game_status,aggregated_rating,aggregated_rating_count,hypes,name_clean,sales_platform
0,376092,2025-10-31,Single player,Indie,Silly Survivors,PC (Microsoft Windows),100.0,"Silly Survivors - a casual, pared-down rogueli...",Action,100.0,6,0,NaN,NaN,NaN,NaN,NaN,NaN,silly_survivors,PC
1,330539,2025-04-18,"Multiplayer, Co-operative, Massively Multiplay...","Shooter, Role-playing (RPG), Simulator",Oxide: Survival Island,Android,100.0,OXIDE is a mobile multiplayer survival simulat...,"Survival, Stealth, Sandbox, Open world",100.0,9,0,"Here you are, alone on the abandoned island, w...",NaN,NaN,NaN,NaN,NaN,oxide_survival_island,And
2,330539,2025-04-18,"Multiplayer, Co-operative, Massively Multiplay...","Shooter, Role-playing (RPG), Simulator",Oxide: Survival Island,iOS,100.0,OXIDE is a mobile multiplayer survival simulat...,"Survival, Stealth, Sandbox, Open world",100.0,9,0,"Here you are, alone on the abandoned island, w...",NaN,NaN,NaN,NaN,NaN,oxide_survival_island,iOS
3,328386,2025-01-16,NaN,NaN,Rooftop Rascal: The Claus Cat,PlayStation 4,100.0,"In ""Rooftop Rascal: The Claus Cat,"" take contr...",NaN,100.0,7,0,NaN,[196045],NaN,NaN,NaN,NaN,rooftop_rascal_the_claus_cat,PS4
4,328386,2025-01-16,NaN,NaN,Rooftop Rascal: The Claus Cat,PlayStation 5,100.0,"In ""Rooftop Rascal: The Claus Cat,"" take contr...",NaN,100.0,7,0,NaN,[196045],NaN,NaN,NaN,NaN,rooftop_rascal_the_claus_cat,PS5


In [505]:
# Приоседеним игры с конкретными платформами 
games_sales_platform = games_ext.merge(sales_not_all, left_on=['name_clean', 'sales_platform'], right_on=['name_clean', 'console'], how='inner', suffixes=('_games', '_sales'))
games_sales_platform.drop(columns = ['img', 'title', 'console', 'developer', 'release_date', 'last_update'], inplace=True)
games_sales_platform.info()

<class 'pandas.DataFrame'>
RangeIndex: 14373 entries, 0 to 14372
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       14373 non-null  int64  
 1   first_release_date       14368 non-null  str    
 2   game_modes               14243 non-null  str    
 3   genres                   14337 non-null  str    
 4   name                     14373 non-null  str    
 5   platforms                14373 non-null  str    
 6   rating                   14259 non-null  float64
 7   summary                  14359 non-null  str    
 8   themes                   13566 non-null  str    
 9   total_rating             14373 non-null  float64
 10  total_rating_count       14373 non-null  int64  
 11  game_type                14373 non-null  int64  
 12  storyline                5885 non-null   str    
 13  age_ratings              13601 non-null  str    
 14  game_status              291 non-

Ура еще 14к игр

<!-- ### Собираем все что намерджилосб -->

In [506]:
games_sales_all.columns

Index(['id', 'first_release_date', 'game_modes', 'genres', 'name', 'platforms',
       'rating', 'summary', 'themes', 'total_rating', 'total_rating_count',
       'game_type', 'storyline', 'age_ratings', 'game_status',
       'aggregated_rating', 'aggregated_rating_count', 'hypes', 'name_clean',
       'genre', 'publisher', 'vg_score', 'critic_score', 'user_score',
       'total_shipped', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales',
       'other_sales'],
      dtype='str')

In [507]:
games_sales_platform.columns

Index(['id', 'first_release_date', 'game_modes', 'genres', 'name', 'platforms',
       'rating', 'summary', 'themes', 'total_rating', 'total_rating_count',
       'game_type', 'storyline', 'age_ratings', 'game_status',
       'aggregated_rating', 'aggregated_rating_count', 'hypes', 'name_clean',
       'sales_platform', 'genre', 'publisher', 'vg_score', 'critic_score',
       'user_score', 'total_shipped', 'total_sales', 'na_sales', 'jp_sales',
       'pal_sales', 'other_sales'],
      dtype='str')

In [508]:
games_with_sales = pd.concat([games_sales_all, games_sales_platform], ignore_index=True)
games_with_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 15319 entries, 0 to 15318
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       15319 non-null  int64  
 1   first_release_date       15313 non-null  str    
 2   game_modes               15186 non-null  str    
 3   genres                   15279 non-null  str    
 4   name                     15319 non-null  str    
 5   platforms                15319 non-null  object 
 6   rating                   15198 non-null  float64
 7   summary                  15304 non-null  str    
 8   themes                   14476 non-null  str    
 9   total_rating             15319 non-null  float64
 10  total_rating_count       15319 non-null  int64  
 11  game_type                15319 non-null  int64  
 12  storyline                6357 non-null   str    
 13  age_ratings              14507 non-null  str    
 14  game_status              315 non-

In [509]:
# Уберем ненуднжые колоночки
games_with_sales.drop(columns=['id', 'name_clean', 'total_rating_count', 'game_status', 'aggregated_rating', 'aggregated_rating', 'aggregated_rating_count', 'vg_score', 'critic_score', 'user_score', 'total_shipped', 'genre', 'publisher'], inplace=True)
games_with_sales.head()


,first_release_date,game_modes,genres,name,platforms,rating,summary,themes,total_rating,game_type,storyline,age_ratings,hypes,total_sales,na_sales,jp_sales,pal_sales,other_sales,sales_platform
0,2022-04-12,"Single player, Multiplayer, Co-operative","Shooter, Racing, Adventure",Grand Theft Auto V,"[Xbox Series X|S, PlayStation 5]",99.596210,A new generation of Grand Theft Auto V featuri...,"Action, Comedy, Open world",99.596210,3,NaN,"[211581, 211580, 126201, 108449, 126200, 83487...",3.0,220.0,NaN,NaN,NaN,NaN,NaN
1,1999-11-02,Single player,Role-playing (RPG),Chrono Trigger,"[PlayStation 3, PlayStation, PlayStation Porta...",98.599448,An enhanced port of Chrono Trigger that adds a...,"Action, Fantasy, Science fiction",98.599448,10,This is the fateful story of those who discove...,"[163482, 163481]",NaN,5.0,NaN,NaN,NaN,NaN,NaN
2,2001-06-22,Single player,"Puzzle, Adventure",Alone in the Dark: The New Nightmare,"[PC (Microsoft Windows), Dreamcast, PlayStatio...",98.368902,"The Dreamcast, PC and PlayStation 2 ports of A...","Action, Horror, Survival, Mystery",98.368902,11,"Set on October 31, 2001, Edward Carnby’s best ...","[172145, 172148, 172147, 172146]",NaN,1.4,NaN,NaN,NaN,NaN,NaN
3,2022-02-25,"Single player, Multiplayer, Co-operative","Role-playing (RPG), Adventure",Elden Ring,"[Xbox Series X|S, PlayStation 4, Nintendo Swit...",93.576279,Elden Ring is an action RPG developed by FromS...,"Action, Fantasy, Open world",95.238140,0,"Elden Ring takes place in the Lands Between, a...","[57167, 57169, 213790, 57164, 57168, 57165, 57...",96.0,30.0,NaN,NaN,NaN,NaN,NaN
4,2024-08-07,Single player,"Adventure, Visual Novel",Fate/Stay Night Remastered,"[PC (Microsoft Windows), Nintendo Switch]",95.161290,"A remastered version of Fate/Stay Night, a fan...",Fantasy,95.161290,9,The Holy Grail War is a ritual held in Fuyuki ...,"[183731, 183349]",4.0,0.1,NaN,NaN,NaN,NaN,NaN


In [510]:
games_with_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 15319 entries, 0 to 15318
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   first_release_date  15313 non-null  str    
 1   game_modes          15186 non-null  str    
 2   genres              15279 non-null  str    
 3   name                15319 non-null  str    
 4   platforms           15319 non-null  object 
 5   rating              15198 non-null  float64
 6   summary             15304 non-null  str    
 7   themes              14476 non-null  str    
 8   total_rating        15319 non-null  float64
 9   game_type           15319 non-null  int64  
 10  storyline           6357 non-null   str    
 11  age_ratings         14507 non-null  str    
 12  hypes               5191 non-null   float64
 13  total_sales         7537 non-null   float64
 14  na_sales            3927 non-null   float64
 15  jp_sales            1776 non-null   float64
 16  pal_sales      

7к игро - платформ на анализ продаж

In [511]:
games_with_sales.to_csv('games_with_sales.csv', index=False)

In [512]:
games_with_sales[games_with_sales['name'].str.contains('God')].head()

,first_release_date,game_modes,genres,name,platforms,rating,summary,themes,total_rating,game_type,storyline,age_ratings,hypes,total_sales,na_sales,jp_sales,pal_sales,other_sales,sales_platform
14,2022-11-09,Single player,"Role-playing (RPG), Hack and slash/Beat 'em up...",God of War Ragnarök,"[PlayStation 4, PC (Microsoft Windows), PlaySt...",90.961843,God of War: Ragnarök is the ninth installment ...,"Action, Fantasy, Open world",92.788614,0,The freezing winds of Fimbulwinter have come t...,"[188434, 105532, 112550, 112549, 112548, 11255...",77.0,15.0,NaN,NaN,NaN,NaN,NaN
489,2024-07-19,Single player,"Strategy, Hack and slash/Beat 'em up",Kunitsu-Gami: Path of the Goddess,"[Xbox Series X|S, PlayStation 4, Nintendo Swit...",74.418831,Fend off foul creatures and lead the Spirit St...,"Action, Fantasy",79.909415,0,NaN,"[181634, 181632, 181633, 181636, 222091, 17756...",20.0,NaN,NaN,NaN,NaN,NaN,NaN
533,2013-11-14,"Single player, Co-operative","Shooter, Role-playing (RPG), Adventure",God Eater 2,"[PlayStation Vita, PlayStation Portable]",79.272552,God Eater 2 is an action role-playing game and...,Action,79.272552,0,NaN,NaN,NaN,0.7,NaN,NaN,NaN,NaN,NaN
803,2006-03-21,Single player,"Shooter, Adventure",The Godfather,"[Xbox, PC (Microsoft Windows), PlayStation 2]",67.792612,The Godfather puts you into the action of the ...,"Action, Historical, Drama, Open world",73.896306,0,"In Little Italy in 1936, a young Aldo Trapani ...","[72920, 54405, 54403, 54402, 54404, 184466, 54...",NaN,1.0,NaN,NaN,NaN,NaN,NaN
887,1991-12-31,Single player,Shooter,The Godfather,[Amiga],NaN,"Action game by U.S. Gold, released in conjunct...","Action, Open world",71.800000,0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
